# 🌾 Rendement agricole — RelativeCropyield (CLIMADA)

Exploration du module `RelativeCropyield` de CLIMADA pour modéliser l'impact du changement climatique sur les cultures.

**Aléa** : Anomalie de rendement (RC)  
**Zone** : France / Europe  
**Données** : ISIMIP crop models / synthétique  
**Cultures** : Blé, maïs, riz, soja

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

from climada.hazard import Hazard, Centroids
from climada.entity import Exposures, ImpactFuncSet, ImpactFunc
from climada.engine import Impact

try:
    from climada.hazard import RelativeCropyield
    from climada.entity.impact_funcs.relative_cropyield import ImpfRelativeCropyield
    PETALS_OK = True
except ImportError:
    PETALS_OK = False
    print("⚠️ climada_petals non disponible — on utilisera des données synthétiques")

print(f"CLIMADA petals disponible : {PETALS_OK}")

## 1. Aléa — Anomalies de rendement agricole

Le module `RelativeCropyield` utilise les sorties des modèles de culture ISIMIP (Inter-Sectoral Impact Model Intercomparison Project) :
- **Modèles agricoles** : GEPIC, LPJmL, PEPIC, etc.
- **Modèles climatiques** : GFDL-ESM4, IPSL-CM6A-LR, MPI-ESM1-2-HR, UKESM1-0-LL
- **Cultures** : blé (wheat), maïs (maize), riz (rice), soja (soy)
- **Scénarios** : historique, SSP126, SSP370, SSP585

La variable est le **rendement relatif** : ratio par rapport à une période de référence (1 = normal, <1 = perte, >1 = gain).

In [ ]:
# --- CLIMADA RelativeCropyield (nécessite données ISIMIP) ---
# from climada.hazard import RelativeCropyield
# crop_haz = RelativeCropyield.from_isimip_netcdf(
#     input_dir='/path/to/isimip/',
#     bbox=(-5, 42, 10, 51),  # France
#     yearrange=(1980, 2020),
#     ag_model='gepic',
#     cl_model='gfdl-esm4',
#     scenario='historical',
#     crop='wheat',
#     irr='noirr'
# )

# --- Données synthétiques : rendement relatif annuel ---
from scipy.sparse import csr_matrix, vstack

np.random.seed(42)

# Grille France : 42-51°N, -5 à 10°E
lon = np.arange(-5, 10.5, 0.5)
lat = np.arange(42, 51.5, 0.5)
lon_grid, lat_grid = np.meshgrid(lon, lat)
lon_flat = lon_grid.flatten()
lat_flat = lat_grid.flatten()
n_centroids = len(lon_flat)

centroids = Centroids.from_lat_lon(lat_flat, lon_flat)
print(f"Centroids : {centroids.size} points")

# 4 cultures × 40 ans (1981-2020)
crops = ['wheat', 'maize', 'rice', 'soy']
n_years = 40
years = np.arange(1981, 1981 + n_years)

crop_hazards = {}

for crop in crops:
    intensity_list = []
    frequency = np.ones(n_years) / n_years
    
    # Paramètres par culture
    if crop == 'wheat':
        base_yield = 1.0
        trend = -0.003  # léger déclin
        volatility = 0.15
    elif crop == 'maize':
        base_yield = 1.0
        trend = -0.005  # plus sensible à la chaleur
        volatility = 0.20
    elif crop == 'rice':
        base_yield = 1.0
        trend = -0.002
        volatility = 0.12
    else:  # soy
        base_yield = 1.0
        trend = -0.004
        volatility = 0.18
    
    for y in range(n_years):
        # Rendement de base + tendance + variabilité
        rel_yield = base_yield + trend * y + np.random.normal(0, volatility, n_centroids)
        
        # Gradient spatial : sud plus affecté par la chaleur
        lat_effect = (lat_flat - 51) / (42 - 51) * (-0.1) * (y / n_years)
        rel_yield += lat_effect
        
        # Années de crise (2003, 2018, 2022 analogues)
        crisis_years = [22, 37, 39]  # indices
        if y in crisis_years:
            rel_yield -= np.random.uniform(0.1, 0.3, n_centroids)
        
        # L'intensité dans CLIMADA = 1 - rendement relatif (anomalie négative)
        # Valeur positive = perte de rendement
        anomaly = np.maximum(1 - rel_yield, 0)
        intensity_list.append(csr_matrix(anomaly))
    
    intensity_matrix = vstack(intensity_list)
    
    crop_haz = Hazard(
        haz_type='RC',
        centroids=centroids,
        intensity=intensity_matrix,
        frequency=frequency,
        event_id=np.arange(1, n_years + 1),
        event_name=[f'{crop}_{y}' for y in years],
        date=pd.date_range('1981-07-01', periods=n_years, freq='365D').to_numpy().astype('datetime64[ns]').astype(int) // 10**9,
        units='relative yield loss'
    )
    crop_hazards[crop] = crop_haz
    print(f"{crop:6s} — max loss: {crop_haz.intensity.max():.3f}, mean: {crop_haz.intensity.mean():.4f}")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

for idx, (crop, haz) in enumerate(crop_hazards.items()):
    ax = axes[idx // 2, idx % 2]
    
    # Anomalie moyenne par année
    mean_loss = np.array(haz.intensity.mean(axis=1)).flatten()
    
    colors = ['green' if v < 0.05 else 'orange' if v < 0.15 else 'red' for v in mean_loss]
    ax.bar(years, mean_loss, color=colors, alpha=0.8)
    
    z = np.polyfit(years, mean_loss, 1)
    ax.plot(years, np.polyval(z, years), 'k--', linewidth=2, 
            label=f'Tendance : {z[0]:+.5f}/an')
    
    ax.set_xlabel('Année')
    ax.set_ylabel('Perte relative moyenne')
    ax.set_title(f'{crop.capitalize()} — Anomalie de rendement')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 2. Carte spatiale des anomalies

Distribution géographique de la perte de rendement moyenne et lors d'une année de crise.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

for idx, (crop, haz) in enumerate(crop_hazards.items()):
    ax = axes[idx // 2, idx % 2]
    
    # Anomalie moyenne sur toute la période
    mean_spatial = np.array(haz.intensity.mean(axis=0)).flatten()
    
    sc = ax.scatter(lon_flat, lat_flat, c=mean_spatial, s=20, cmap='RdYlGn_r', vmin=0)
    ax.set_title(f'{crop.capitalize()} — Perte moyenne (1981-2020)')
    ax.set_xlabel('Longitude')
    if idx % 2 == 0:
        ax.set_ylabel('Latitude')
    plt.colorbar(sc, ax=ax, label='Perte relative', shrink=0.8)

plt.tight_layout()
plt.show()

## 3. Exposition — Production agricole

L'exposition utilise la valeur de production agricole par cellule. CLIMADA propose :
- `CropProduction` : données ISIMIP de production (tonnes × prix)
- `SpamAgrar` : Spatial Production Allocation Model (Harvard Dataverse)

In [ ]:
from shapely.geometry import Point
import geopandas as gpd

# --- CropProduction CLIMADA (nécessite données) ---
# from climada.entity import CropProduction
# crop_exp = CropProduction.from_isimip_netcdf(
#     input_dir='/path/to/isimip/',
#     bbox=(-5, 42, 10, 51),
#     yearrange=(2000, 2020),
#     crop='wheat',
#     irr='noirr'
# )

# --- Exposition synthétique : zones agricoles françaises ---
n_assets = 300
np.random.seed(101)

# Grandes régions agricoles
agri_regions = {
    'Beauce': (1.5, 48.0, 0.20, 'wheat'),
    'Champagne': (3.5, 48.8, 0.12, 'wheat'),
    'Aquitaine': (0.0, 44.5, 0.15, 'maize'),
    'Alsace': (7.5, 48.5, 0.08, 'maize'),
    'Camargue': (4.5, 43.5, 0.05, 'rice'),
    'Bourgogne': (3.5, 47.0, 0.10, 'wheat'),
    'Picardie': (2.5, 49.8, 0.10, 'wheat'),
    'Poitou': (-0.3, 46.5, 0.10, 'maize'),
    'Midi_Pyrenees': (1.5, 43.5, 0.10, 'soy'),
}

lons, lats, values, crop_types = [], [], [], []
for region, (clon, clat, weight, crop) in agri_regions.items():
    n = int(n_assets * weight)
    lons.extend(np.random.normal(clon, 0.5, n))
    lats.extend(np.random.normal(clat, 0.3, n))
    # Valeur de production (EUR/ha × surface)
    values.extend(np.random.exponential(500_000, n))
    crop_types.extend([crop] * n)

remaining = n_assets - len(lons)
if remaining > 0:
    lons.extend(np.random.uniform(-3, 8, remaining))
    lats.extend(np.random.uniform(43, 50, remaining))
    values.extend(np.random.exponential(300_000, remaining))
    crop_types.extend(['wheat'] * remaining)

gdf = gpd.GeoDataFrame({
    'value': np.array(values[:n_assets]),
    'impf_RC': np.ones(n_assets, dtype=int),
    'crop': crop_types[:n_assets],
    'geometry': [Point(lo, la) for lo, la in zip(lons[:n_assets], lats[:n_assets])]
}, crs='EPSG:4326')

exposure = Exposures(gdf)
exposure.check()

# Stats par culture
for crop in crops:
    mask = exposure.gdf['crop'] == crop
    print(f"{crop:6s} : {mask.sum():3d} actifs, valeur = {exposure.gdf.loc[mask, 'value'].sum():>12,.0f} USD")
print(f"Total  : {len(exposure.gdf)} actifs, valeur = {exposure.gdf['value'].sum():>12,.0f} USD")

## 4. Vulnérabilité — Fonction d'impact rendement

La fonction d'impact pour les rendements est directe : la perte relative de rendement **est** le dommage.

$$MDR = \min(\text{yield\_loss}, 1.0)$$

CLIMADA propose `ImpfRelativeCropyield.impf_relativeyield()` — une fonction linéaire 1:1.

In [ ]:
intensity_range = np.arange(0, 1.01, 0.01)

# Linéaire 1:1 (CLIMADA default)
mdr_linear = intensity_range.copy()

# Avec seuil de franchise (les petites variations ne comptent pas)
threshold = 0.1
mdr_threshold = np.where(intensity_range > threshold,
                          (intensity_range - threshold) / (1 - threshold), 0)

# Non-linéaire : pertes faibles sous-estimées, pertes fortes amplifiées
mdr_nonlinear = intensity_range ** 1.5

impf_linear = ImpactFunc(
    id=1, haz_type='RC',
    intensity=intensity_range,
    mdd=mdr_linear,
    paa=np.ones_like(mdr_linear),
    intensity_unit='relative yield loss',
    name='Linéaire 1:1'
)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(intensity_range, mdr_linear, 'b-', linewidth=2, label='Linéaire 1:1 (CLIMADA)')
ax.plot(intensity_range, mdr_threshold, 'r--', linewidth=2, label=f'Avec franchise ({threshold:.0%})')
ax.plot(intensity_range, mdr_nonlinear, 'g:', linewidth=2, label='Non-linéaire (x^1.5)')
ax.plot([0, 1], [0, 1], 'k:', alpha=0.3)
ax.set_xlabel('Perte de rendement relative')
ax.set_ylabel('MDR (damage ratio)')
ax.set_title('Fonctions de vulnérabilité — Rendement agricole')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Impact — Calcul des pertes par culture

On calcule l'impact pour chaque culture séparément, puis on agrège.

In [ ]:
impf_set = ImpactFuncSet([impf_linear])

crop_results = {}
for crop in crops:
    haz = crop_hazards[crop]
    
    # Filtrer l'exposition pour cette culture
    mask = exposure.gdf['crop'] == crop
    if mask.sum() == 0:
        continue
    
    crop_exp = Exposures(exposure.gdf[mask].copy())
    crop_exp.check()
    
    imp = Impact()
    imp.calc(crop_exp, impf_set, haz, save_mat=True)
    
    crop_results[crop] = {
        'eai': imp.aai_agg,
        'at_event': imp.at_event,
        'freq_curve': imp.calc_freq_curve(),
        'impact': imp,
        'n_assets': mask.sum()
    }
    
    print(f"{crop:6s} — EAI: {imp.aai_agg:>10,.0f} USD | Max: {imp.at_event.max():>10,.0f} USD | Assets: {mask.sum()}")

total_eai = sum(r['eai'] for r in crop_results.values())
print(f"\n{'TOTAL':6s} — EAI: {total_eai:>10,.0f} USD")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

crop_colors = {'wheat': '#DAA520', 'maize': '#228B22', 'rice': '#4682B4', 'soy': '#8B4513'}

# 1. EAI par culture
eai_vals = {c: r['eai'] for c, r in crop_results.items()}
bars = axes[0, 0].bar(eai_vals.keys(), [v / 1e6 for v in eai_vals.values()],
                       color=[crop_colors[c] for c in eai_vals.keys()], alpha=0.8)
axes[0, 0].set_ylabel('EAI (M USD/an)')
axes[0, 0].set_title('Impact annuel moyen par culture')

# 2. Courbes de fréquence
for crop, res in crop_results.items():
    fc = res['freq_curve']
    axes[0, 1].plot(fc.return_per, fc.impact / 1e6, linewidth=2, 
                     color=crop_colors[crop], label=crop.capitalize())
axes[0, 1].set_xlabel('Période de retour (ans)')
axes[0, 1].set_ylabel('Perte (M USD)')
axes[0, 1].set_title('Courbes de fréquence par culture')
axes[0, 1].set_xscale('log')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# 3. Pertes temporelles empilées
bottom = np.zeros(n_years)
for crop, res in crop_results.items():
    axes[1, 0].bar(years, res['at_event'] / 1e6, bottom=bottom,
                    color=crop_colors[crop], alpha=0.8, label=crop.capitalize())
    bottom += res['at_event'] / 1e6
axes[1, 0].set_xlabel('Année')
axes[1, 0].set_ylabel('Perte (M USD)')
axes[1, 0].set_title('Pertes annuelles par culture (empilées)')
axes[1, 0].legend()

# 4. Part de chaque culture dans l'EAI
labels = [c.capitalize() for c in crop_results.keys()]
sizes = [r['eai'] for r in crop_results.values()]
axes[1, 1].pie(sizes, labels=labels, autopct='%1.1f%%',
               colors=[crop_colors[c] for c in crop_results.keys()], startangle=90)
axes[1, 1].set_title('Répartition de l\'EAI par culture')

plt.tight_layout()
plt.show()

## 6. Scénarios climatiques

Comparaison des pertes sous différents scénarios SSP pour le blé.

In [ ]:
# Générer des projections simplifiées pour le blé
scenarios = {
    'Historique': 0.0,
    'SSP1-2.6': -0.02,   # léger déclin
    'SSP3-7.0': -0.06,   # déclin marqué
    'SSP5-8.5': -0.10,   # déclin sévère
}

scenario_results = {}
future_years = np.arange(2021, 2061)
n_fut = len(future_years)

for scenario, extra_trend in scenarios.items():
    intensity_list = []
    for y in range(n_fut):
        rel_yield = 1.0 + (-0.003 + extra_trend / n_fut * y) * y
        noise = np.random.normal(0, 0.15, n_centroids)
        lat_effect = (lat_flat - 51) / (42 - 51) * (-0.1 + extra_trend) * (y / n_fut)
        anomaly = np.maximum(1 - (rel_yield + noise + lat_effect), 0)
        intensity_list.append(csr_matrix(anomaly))
    
    haz_fut = Hazard(
        haz_type='RC',
        centroids=centroids,
        intensity=vstack(intensity_list),
        frequency=np.ones(n_fut) / n_fut,
        event_id=np.arange(1, n_fut + 1),
        event_name=[f'wheat_{y}' for y in future_years],
        date=pd.date_range('2021-07-01', periods=n_fut, freq='365D').to_numpy().astype('datetime64[ns]').astype(int) // 10**9,
        units='relative yield loss'
    )
    
    wheat_mask = exposure.gdf['crop'] == 'wheat'
    wheat_exp = Exposures(exposure.gdf[wheat_mask].copy())
    wheat_exp.check()
    
    imp = Impact()
    imp.calc(wheat_exp, impf_set, haz_fut, save_mat=True)
    
    scenario_results[scenario] = {
        'eai': imp.aai_agg,
        'at_event': imp.at_event,
        'years': future_years
    }
    print(f"{scenario:12s} — EAI blé : {imp.aai_agg:>10,.0f} USD")

# Visualisation
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ssp_colors = {'Historique': 'gray', 'SSP1-2.6': 'blue', 'SSP3-7.0': 'orange', 'SSP5-8.5': 'red'}

for scenario, res in scenario_results.items():
    # Moyenne glissante 5 ans
    losses = pd.Series(res['at_event']).rolling(5, min_periods=1).mean()
    axes[0].plot(res['years'], losses / 1e6, linewidth=2, 
                 color=ssp_colors[scenario], label=scenario)
axes[0].set_xlabel('Année')
axes[0].set_ylabel('Perte (M USD, moy. glissante 5 ans)')
axes[0].set_title('Projections blé — Scénarios SSP')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# EAI bar chart
bars = axes[1].bar(scenario_results.keys(), 
                    [r['eai'] / 1e6 for r in scenario_results.values()],
                    color=[ssp_colors[s] for s in scenario_results.keys()], alpha=0.8)
axes[1].set_ylabel('EAI (M USD/an)')
axes[1].set_title('Impact annuel moyen — Blé par scénario')
axes[1].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.show()

## 7. Synthèse

| Aspect | Détail |
|--------|--------|
| Source données | ISIMIP (modèles de culture GEPIC, LPJmL, PEPIC) |
| Variable | Rendement relatif (ratio vs référence) |
| Cultures | Blé, maïs, riz, soja |
| Résolution | 0.5° (~50 km) |
| Exposition | CropProduction, SpamAgrar (Harvard) |

### Points clés
- Le maïs est la culture la plus vulnérable au réchauffement en France (stress thermique)
- Le gradient nord-sud s'amplifie avec le changement climatique
- Sous SSP5-8.5, les pertes de rendement moyennes augmentent de 3-5× d'ici 2060
- La diversification des cultures est une stratégie d'adaptation clé

### Prochaines étapes
- Télécharger les données ISIMIP réelles (multi-modèle, multi-culture)
- Coupler avec CropProduction/SpamAgrar pour une exposition réaliste
- Analyser l'impact de l'irrigation (`irr` vs `noirr`)
- Explorer les mesures d'adaptation (variétés résistantes, irrigation)